In [48]:
import pandas as pd
import numpy as np

In [49]:
df = pd.read_csv("HousePricePrediction.csv")

In [50]:
df.info()
print(df.isna().sum())
df["MSZoning"] = df["MSZoning"].fillna(df["MSZoning"].mode()[0])
df["Exterior1st"] = df["Exterior1st"].fillna(df["Exterior1st"].mode()[0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            2919 non-null   int64  
 1   MSSubClass    2919 non-null   int64  
 2   MSZoning      2915 non-null   object 
 3   LotArea       2919 non-null   int64  
 4   LotConfig     2919 non-null   object 
 5   BldgType      2919 non-null   object 
 6   OverallCond   2919 non-null   int64  
 7   YearBuilt     2919 non-null   int64  
 8   YearRemodAdd  2919 non-null   int64  
 9   Exterior1st   2918 non-null   object 
 10  BsmtFinSF2    2918 non-null   float64
 11  TotalBsmtSF   2918 non-null   float64
 12  SalePrice     1460 non-null   float64
dtypes: float64(3), int64(6), object(4)
memory usage: 296.6+ KB
Id                 0
MSSubClass         0
MSZoning           4
LotArea            0
LotConfig          0
BldgType           0
OverallCond        0
YearBuilt          0
YearRemodAdd    

In [51]:
df["BsmtFinSF2"] = pd.to_numeric(df["BsmtFinSF2"])
df['BsmtFinSF2'] = df["BsmtFinSF2"].fillna(df["BsmtFinSF2"].mode()[0])

df["TotalBsmtSF"] = pd.to_numeric(df["TotalBsmtSF"])
df['TotalBsmtSF'] = df["TotalBsmtSF"].fillna(df["TotalBsmtSF"].mode()[0])

df["SalePrice"] = df["SalePrice"].fillna(df["SalePrice"].median())

print(df.isna().sum())

Id              0
MSSubClass      0
MSZoning        0
LotArea         0
LotConfig       0
BldgType        0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
Exterior1st     0
BsmtFinSF2      0
TotalBsmtSF     0
SalePrice       0
dtype: int64


In [52]:
if 'Id' in df.columns:
    df.drop("Id", axis=1, inplace=True)

encoded = pd.get_dummies(df, columns=["MSZoning", "BldgType", "LotConfig", "Exterior1st"], drop_first=False)

print(encoded)

      MSSubClass  LotArea  OverallCond  YearBuilt  YearRemodAdd  BsmtFinSF2  \
0             60     8450            5       2003          2003         0.0   
1             20     9600            8       1976          1976         0.0   
2             60    11250            5       2001          2002         0.0   
3             70     9550            5       1915          1970         0.0   
4             60    14260            5       2000          2000         0.0   
...          ...      ...          ...        ...           ...         ...   
2914         160     1936            7       1970          1970         0.0   
2915         160     1894            5       1970          1970         0.0   
2916          20    20000            7       1960          1996         0.0   
2917          85    10441            5       1992          1992         0.0   
2918          60     9627            5       1993          1994         0.0   

      TotalBsmtSF  SalePrice  MSZoning_C (all)  MSZ

In [53]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = encoded.drop('SalePrice', axis=1)
y = encoded['SalePrice']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [54]:
# Display the shapes of the training and testing sets
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (2335, 37)
Shape of X_test: (584, 37)
Shape of y_train: (2335,)
Shape of y_test: (584,)


In [55]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
linear_model = LinearRegression()

# Train the model using the training data
linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [56]:
# Use the trained model to predict on the test data
y_pred = linear_model.predict(X_test)

# Display the first few predictions
print("First 5 predictions:")
print(y_pred[:5])

First 5 predictions:
[159782.38352728 175725.49224753 142726.50001166 179656.50319192
 192031.41956295]


In [57]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)

# Calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Calculate R-squared (R2) Score
r2 = r2_score(y_test, y_pred)

# Display the evaluation metrics
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R-squared (R2) Score: {r2}")

Mean Absolute Error (MAE): 32043.26653400237
Mean Squared Error (MSE): 2442727069.6652675
Root Mean Squared Error (RMSE): 49423.95238814139
R-squared (R2) Score: 0.3367956470470017


## Model Performance Analysis

Let's analyze the performance of the Linear Regression model based on the calculated metrics:

*   **R-squared (R2) Score:** `0.3368`
    *   **Analysis:** This is the most crucial metric for understanding model performance in regression. An R² of 0.3368 means that approximately 33.68% of the variance in house prices can be explained by our model's features. This indicates that the model is not explaining a large proportion of the variability in the sale prices, suggesting a relatively weak fit. There's a lot of unexplained variance.

*   **Mean Absolute Error (MAE):** `32043.27`
    *   **Analysis:** On average, our model's predictions are off by about $32,043.27 from the actual sale price. This gives a straightforward interpretation of the average error magnitude.

*   **Mean Squared Error (MSE):** `2,442,727,069.67`
    *   **Analysis:** MSE penalizes larger errors more heavily. The high value suggests there are some substantial individual prediction errors.

*   **Root Mean Squared Error (RMSE):** `49423.95`
    *   **Analysis:** RMSE is the square root of MSE and is in the same units as the target variable (SalePrice). An RMSE of $49,423.95 means that the typical difference between our model's predicted values and the actual observed values is around this amount.

**Overall Conclusion:**
While the model has been trained, an R² score of 0.3368 suggests that its predictive power is quite limited. The MAE and RMSE values also indicate that the predictions have a significant average error margin. This model would likely not be considered very good for accurate house price prediction.

**Possible Next Steps to Improve Performance:**
1.  **Feature Engineering:** Create new features from existing ones (e.g., age of house, total square footage by combining basement and above-ground SF). More advanced feature engineering might capture more complex relationships.
2.  **Handle More Missing Values:** Review the initial dataset for other columns with missing values that were not addressed, or consider more sophisticated imputation methods.
3.  **Explore Other Categorical Variables:** There might be other object-type columns that are still in `df` that need to be one-hot encoded or handled differently.
4.  **Polynomial Features:** For some numerical features, non-linear relationships might exist that a linear model cannot capture.
5.  **Different Models:** Try more complex models like RandomForest Regressor, Gradient Boosting Regressor (e.g., XGBoost, LightGBM), or even neural networks, which can often capture non-linear relationships better.
6.  **Outlier Detection:** Outliers in the data can heavily influence linear models. Identifying and handling them could improve performance.